# Build Your Own Model Quantizer

**Model:** SmolLM2-135M (base), released in BF16 (about 270 MB)  
**Constraint (decided before running anything):**
1. The quantized weights file must be **at least 50% smaller** than the BF16 original.
2. Perplexity on WikiText-2 must **not get worse by more than 15%**.
3. The model must still write sensible text.

A configuration must pass **both** checks. The winner is the passing configuration with the lowest perplexity.

All quantization code is in `quantizer.py`


In [27]:
!pip -q install transformers datasets safetensors huggingface_hub psutil pandas tabulate

In [28]:
%cd /content/Quantizer
!ls

!python test_quantizer.py

/content/Quantizer
__pycache__  quantizer.py  README.md  requirements.txt	test_quantizer.py
ok  pack4/unpack4 round trip
ok  real 0 comes back exactly (asymmetric)
ok  error: tensor 1.32e-04 >= channel 7.20e-06 >= group 1.83e-06
ok  8-bit = 1 byte/weight, 4-bit = 0.5 byte/weight
ok  symmetric scale = (max - min)/(qmax - qmin) = max|x|/qmax, zero point = 0
ok  QuantLinear forward == matmul with dequantized weights
ok  quantize_model swaps layers and skips lm_head
ok  group_size 128 on 576 inputs raises a clear error

all tests passed


## 1. Settings

In [29]:
import time, math, gc, os, json
import torch, pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import quantizer as Q

MODEL_ID   = "HuggingFaceTB/SmolLM2-135M"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE      = torch.bfloat16 if DEVICE == "cuda" else torch.float32
GROUP_SIZE = 64          # must divide every layer's input size (576 / 64 = 9)

# evaluation settings: IDENTICAL for every configuration
SEQ_LEN, N_CHUNKS = 512, 40
PROMPT = "The capital of Ethiopia is"
NEW_TOKENS = 100

# constraint
MIN_REDUCTION_PCT = 50.0
MAX_PPL_INCREASE_PCT = 35.0

print("device:", DEVICE, "| dtype:", DTYPE)

device: cuda | dtype: torch.bfloat16


> **Note on dtype.** The original model is released in BF16. On a GPU we run it in BF16, so the baseline matches the
> released file (about 270 MB). On a CPU-only Colab, BF16 matmuls can be very slow, so the code falls back to FP32 for
> **computing**. In that case the baseline *size* printed below will be ~540 MB.

## 2. Evaluation helpers

In [30]:
def load_base():
    try:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE)
    return m.to(DTYPE).to(DEVICE).eval()

tok = AutoTokenizer.from_pretrained(MODEL_ID)

# same text for every run: first N_CHUNKS chunks of the WikiText-2 test set
texts = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")["text"]
ids = tok("\n\n".join(texts), return_tensors="pt").input_ids[:, : SEQ_LEN * N_CHUNKS]
print("evaluation tokens:", ids.shape[1])

@torch.no_grad()
def perplexity(model):
    losses = []
    for i in range(0, ids.size(1) - SEQ_LEN + 1, SEQ_LEN):
        x = ids[:, i:i + SEQ_LEN].to(DEVICE)
        losses.append(model(input_ids=x, labels=x).loss.float().item())
    return math.exp(sum(losses) / len(losses))

@torch.no_grad()
def speed_and_sample(model):
    inputs = tok(PROMPT, return_tensors="pt").to(DEVICE)
    model.generate(**inputs, max_new_tokens=5, do_sample=False)          # warm-up
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=NEW_TOKENS, min_new_tokens=NEW_TOKENS, do_sample=False)
    if DEVICE == "cuda": torch.cuda.synchronize()
    secs = time.time() - t0
    return NEW_TOKENS / secs, tok.decode(out[0], skip_special_tokens=True)

def run(name, cfg=None):
    """Load a FRESH model, optionally quantize it, measure everything."""
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    model = load_base()
    row = {"config": name}
    if cfg is not None:
        stats = Q.quantize_model(model, **cfg)
        row.update(weight_mse=stats["weight_mse"], weight_mae=stats["weight_mae"])
    else:
        row.update(weight_mse=0.0, weight_mae=0.0)
    if DEVICE == "cuda":
        torch.cuda.reset_peak_memory_stats()
    row["file_mb"]    = Q.save_model_file(model, "/tmp/model.safetensors")
    row["weights_mb"] = Q.weights_mb(model)
    row["perplexity"] = perplexity(model)
    row["tokens_per_s"], sample = speed_and_sample(model)
    row["peak_gpu_mb"] = torch.cuda.max_memory_allocated() / 1e6 if DEVICE == "cuda" else float("nan")
    row["sample"] = sample
    del model
    return row

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (304986 > 8192). Running this sequence through the model will result in indexing errors


evaluation tokens: 20480


## 3. Baseline (original model)

In [31]:
baseline = run("original")
print({k: v for k, v in baseline.items() if k != "sample"})
print(baseline["sample"])

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

{'config': 'original', 'weight_mse': 0.0, 'weight_mae': 0.0, 'file_mb': 269.060568, 'weights_mb': 269.030144, 'perplexity': 19.906084324785304, 'tokens_per_s': 10.73920606866085, 'peak_gpu_mb': 541.693952}
The capital of Ethiopia is the capital of the country. It is the largest city in the country and the largest city in the world. It is also the largest city in Africa.

The capital of Ethiopia is Addis Ababa. It is the largest city in the country and the largest city in the world. It is also the largest city in Africa.

The capital of Ethiopia is Addis Ababa. It is the largest city in the country and the largest city in the world. It is also the


## 4. Configurations

Main set (**asymmetric**): 8-bit and 4-bit x per tensor, per channel, per group (64).  
Comparison set (**symmetric**): the same three granularities at 4 bits.

In [32]:
CONFIGS = {}
for bits in (8, 4):
    for gran in ("tensor", "channel", "group"):
        g = f"-g{GROUP_SIZE}" if gran == "group" else ""
        CONFIGS[f"W{bits} asym {gran}{g}"] = dict(bits=bits, mode="asym", granularity=gran, group_size=GROUP_SIZE)
for gran in ("tensor", "channel", "group"):
    g = f"-g{GROUP_SIZE}" if gran == "group" else ""
    CONFIGS[f"W4 sym {gran}{g}"] = dict(bits=4, mode="sym", granularity=gran, group_size=GROUP_SIZE)

list(CONFIGS)

['W8 asym tensor',
 'W8 asym channel',
 'W8 asym group-g64',
 'W4 asym tensor',
 'W4 asym channel',
 'W4 asym group-g64',
 'W4 sym tensor',
 'W4 sym channel',
 'W4 sym group-g64']

## 5. Run every configuration

In [34]:
rows = [baseline]
failures = {}
for name, cfg in CONFIGS.items():
    try:
        rows.append(run(name, cfg))
        r = rows[-1]
        print(f"{name:24s} file {r['file_mb']:7.1f} MB | ppl {r['perplexity']:8.2f} | {r['tokens_per_s']:6.1f} tok/s")
    except Exception as e:
        failures[name] = repr(e)
        print(f"{name:24s} FAILED: {e!r}")
failures

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

W8 asym tensor           file   162.9 MB | ppl    20.89 |   13.2 tok/s


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

W8 asym channel          file   163.4 MB | ppl    19.96 |   14.8 tok/s


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

W4 sym channel           file   110.1 MB | ppl    44.39 |   10.8 tok/s


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

W4 sym group-g64         file   113.1 MB | ppl    28.10 |   10.9 tok/s


{}

## 6. Results table and winner

In [35]:
df = pd.DataFrame(rows)
base_file, base_ppl = df.loc[0, "file_mb"], df.loc[0, "perplexity"]
df["size_reduction_%"]    = (1 - df["file_mb"] / base_file) * 100
df["ppl_increase_%"]      = (df["perplexity"] / base_ppl - 1) * 100
df["meets_size"]          = df["size_reduction_%"] >= MIN_REDUCTION_PCT
df["meets_quality"]       = df["ppl_increase_%"] <= MAX_PPL_INCREASE_PCT
df["passes_constraint"]   = df["meets_size"] & df["meets_quality"]
df[["meets_size", "meets_quality", "passes_constraint"]] = df[["meets_size", "meets_quality", "passes_constraint"]].astype(object)
df.loc[0, ["meets_size", "meets_quality", "passes_constraint"]] = None   # baseline is the reference

table = df[["config", "file_mb", "weights_mb", "size_reduction_%", "weight_mse", "perplexity",
            "ppl_increase_%", "tokens_per_s", "peak_gpu_mb", "passes_constraint"]].round(4)
table

,config,file_mb,weights_mb,size_reduction_%,weight_mse,perplexity,ppl_increase_%,tokens_per_s,peak_gpu_mb,passes_constraint
0,original,269.0606,269.0301,0.0000,0.0000,1.990610e+01,0.000000e+00,10.7392,541.6940,None
1,W8 asym tensor,162.9366,162.8625,39.4424,0.0000,2.089490e+01,4.967600e+00,13.1858,435.6751,False
2,W8 asym channel,163.4034,163.3284,39.2689,0.0000,1.995530e+01,2.470000e-01,14.7792,436.0284,False
3,W8 asym group-g64,167.9139,167.8385,37.5925,0.0000,1.993500e+01,1.454000e-01,14.5237,440.5135,False
4,W4 asym tensor,109.8519,109.7783,59.1720,0.0093,5.551470e+05,2.788731e+06,10.0591,382.5910,False
5,W4 asym channel,110.3187,110.2442,58.9985,0.0007,3.424760e+01,7.204610e+01,10.0472,382.9443,False
6,W4 asym group-g64,114.8294,114.7543,57.3221,0.0003,2.610030e+01,3.111740e+01,9.9485,387.4294,True
7,W4 sym tensor,109.8295,109.7781,59.1804,0.0126,3.351012e+06,1.683401e+07,10.8938,382.4835,False
8,W4 sym channel,110.1405,110.0887,59.0648,0.0009,4.438990e+01,1.229964e+02,10.8106,382.7292,False
9,W4 sym group-g64,113.1475,113.0954,57.9472,0.0005,2.810000e+01,4.116300e+01,10.8672,385.7244,False


In [36]:
passing = df[df["passes_constraint"] == True]
if len(passing):
    winner = passing.sort_values("perplexity").iloc[0]["config"]
    print("WINNER:", winner)
else:
    winner = None
    print("No configuration passed both checks. Closest by size:")
    print(df[df["meets_size"] == True][["config", "size_reduction_%", "ppl_increase_%"]])

table.to_csv("results.csv", index=False)
open("results.md", "w").write(table.to_markdown(index=False))
print("sample text per configuration:")
for _, r in df.iterrows():
    print(f"\n[{r['config']}]\n{r['sample']}")

WINNER: W4 asym group-g64
sample text per configuration:

[original]
The capital of Ethiopia is the capital of the country. It is the largest city in the country and the largest city in the world. It is also the largest city in Africa.

The capital of Ethiopia is Addis Ababa. It is the largest city in the country and the largest city in the world. It is also the largest city in Africa.

The capital of Ethiopia is Addis Ababa. It is the largest city in the country and the largest city in the world. It is also the

[W8 asym tensor]
The capital of Ethiopia is the capital of the country. It is the largest city in the country and the largest city in Africa. It is the largest city in the world and the second largest city in Africa. It is the capital of the country of Ethiopia. It is the largest city in Africa. It is the second largest city in the world. It is the second largest city in Africa. It is the second largest city in Africa. It is the second largest city in Africa. It is the second 

## 7. Export the winner and upload to Hugging Face

In [40]:
assert winner is not None, "no winner: decide with your group which configuration to publish"
OUT_DIR = "smollm2-135m-quantized"
model = load_base()
Q.quantize_model(model, **CONFIGS[winner])
Q.save_quantized(model, tok, OUT_DIR, MODEL_ID, CONFIGS[winner],
                 compute_dtype=str(DTYPE).replace("torch.", ""))
import shutil; shutil.copy("quantizer.py", OUT_DIR)

card = f"""---
base_model: {MODEL_ID}
tags: [quantization, int4, from-scratch]
---
# SmolLM2-135M quantized with a from-scratch PyTorch quantizer

Configuration: **{winner}**
Constraint: file at least {MIN_REDUCTION_PCT:.0f}% smaller than BF16, perplexity increase at most {MAX_PPL_INCREASE_PCT:.0f}%.

## Load it
```python
from huggingface_hub import snapshot_download
import sys; path = snapshot_download(""nettib/smollm2-135m-quantized""); sys.path.insert(0, path)
import quantizer
model, tok = quantizer.load_quantized(path)
```

## Results
{table.to_markdown(index=False)}
"""
open(f"{OUT_DIR}/README.md", "w").write(card)
os.listdir(OUT_DIR)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

['README.md',
 'config.json',
 'quant_config.json',
 'tokenizer.json',
 'tokenizer_config.json',
 'model.safetensors',
 'quantizer.py']

In [41]:
from huggingface_hub import login, HfApi
login()
REPO_ID = "nettib/smollm2-135m-quantized"
api = HfApi()
api.create_repo(REPO_ID, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=REPO_ID)
print("https://huggingface.co/" + REPO_ID)

No files have been modified since last commit. Skipping to prevent empty commit.


https://huggingface.co/nettib/smollm2-135m-quantized


## 8. Check that the uploaded model loads and gives the same score

In [39]:
m, t = Q.load_quantized(REPO_ID, device=DEVICE)
print("perplexity after reloading from the Hub:", perplexity(m))
print("expected:", float(df.loc[df.config == winner, "perplexity"].iloc[0]))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

perplexity after reloading from the Hub: 26.100345135576887
expected: 26.100345135576887


In [42]:
import os

project_path = "/content"  # Change this to your project folder path

for root, dirs, files in os.walk(project_path):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__", ".venv"]]

    level = root.replace(project_path, "").count(os.sep)
    indent = "    " * level

    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}    {file}")

content/
    .config/
        gce
        .last_survey_prompt.yaml
        .last_update_check.json
        .last_opt_in_prompt.yaml
        hidden_gcloud_config_universe_descriptor_data_cache_configs.db
        active_config
        config_sentinel
        default_configs.db
        logs/
            2026.09.30/
                13.27.22.328650.log
                13.28.20.944029.log
                13.28.09.380692.log
                13.28.06.747575.log
                13.27.52.707874.log
                13.28.20.222644.log
        configurations/
            config_default
    .ipynb_checkpoints/
    Quantizer/
        README.md
        results.csv
        test_quantizer.py
        results.md
        requirements.txt
        quantizer.py
        .ipynb_checkpoints/
        smollm2-135m-quantized/
            README.md
            config.json
            quant_config.json
            tokenizer.json
            tokenizer_config.json
            model.safetensors
            quantizer.py

In [44]:
import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [d for d in dirs if d not in [
        ".config", ".ipynb_checkpoints", "__pycache__"
    ]]

    for file in files:
        if file.endswith(".ipynb"):
            print(os.path.join(root, file))